In [131]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Loading data

In [132]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
seasons = pd.read_csv('data/seasons.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')

# Understanding, refining and cleaning data

In [133]:
races = races[races['year'] >= 2020]

In [134]:
races.head()

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1018,1031,2020,1,70,Austrian Grand Prix,2020-07-05,13:10:00,http://en.wikipedia.org/wiki/2020_Austrian_Gra...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1019,1032,2020,2,70,Styrian Grand Prix,2020-07-12,13:10:00,http://en.wikipedia.org/wiki/2020_Styrian_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1020,1033,2020,3,11,Hungarian Grand Prix,2020-07-19,13:10:00,http://en.wikipedia.org/wiki/2020_Hungarian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1021,1034,2020,4,9,British Grand Prix,2020-08-02,13:10:00,http://en.wikipedia.org/wiki/2020_British_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1022,1035,2020,5,9,70th Anniversary Grand Prix,2020-08-09,13:10:00,http://en.wikipedia.org/wiki/70th_Anniversary_...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [135]:
cols = ['raceId', 'circuitId', 'year', 'round', 'name', 'date', 'time']
df = races[cols]

In [136]:
print(df.shape)

(107, 7)


In [137]:
df.dtypes

raceId        int64
circuitId     int64
year          int64
round         int64
name         object
date         object
time         object
dtype: object

In [138]:
qualifying.head()

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


In [139]:
df = df.merge(qualifying, on='raceId', how='left')

In [140]:
df['grid'] = df['position']
df = df.drop(columns='position')

In [141]:
print(df.shape)
df.head()

(2138, 15)


,raceId,circuitId,year,round,name,date,time,qualifyId,driverId,constructorId,number,q1,q2,q3,grid
0,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8378,822,131,77,1:04.111,1:03.015,1:02.939,1
1,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8379,1,131,44,1:04.198,1:03.096,1:02.951,2
2,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8380,830,9,33,1:04.024,1:04.000,1:03.477,3
3,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8381,846,1,4,1:04.606,1:03.819,1:03.626,4
4,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8382,848,9,23,1:04.661,1:03.746,1:03.868,5


1031 is the first race in dataset that we are observing.

In [142]:
driver_standings = driver_standings[driver_standings['raceId'] >= 1031]
constructor_standings = constructor_standings[constructor_standings['raceId'] >= 1031]

In [143]:
driver_standings.head()

,driverStandingsId,raceId,driverId,points,position,positionText,wins
32566,69888,1034,830,52.0,3,3,0
32567,69887,1034,817,20.0,9,9,0
32568,69886,1034,840,20.0,8,8,0
32569,69885,1034,825,1.0,16,16,0
32570,69884,1034,154,0.0,20,20,0


In [144]:
constructor_standings.head()

,constructorStandingsId,raceId,constructorId,points,position,positionText,wins
12316,27522,1034,210,1.0,9,9,0
12317,27521,1034,9,78.0,2,2,0
12318,27520,1034,3,0.0,10,10,0
12319,27519,1034,51,2.0,8,8,0
12320,27518,1034,4,32.0,6,6,0


In [145]:
print('Driver standings shape: ', driver_standings.shape)
print('Constructor standings shape: ', constructor_standings.shape)

Driver standings shape:  (2256, 7)
Constructor standings shape:  (1070, 7)


I am changing column signatures, so I can identify them later easier.

In [146]:
driver_standings = driver_standings.rename(columns={
    'points': 'drv_points',
    'position': 'drv_position',
    'positionText': 'drv_position_text',
    'wins': 'drv_wins',
})

In [147]:
constructor_standings = constructor_standings.rename(columns={
    'points': 'ctor_points',
    'position': 'ctor_position',
    'positionText': 'ctor_position_text',
    'wins': 'ctor_wins',
})

In [148]:
df = df.merge(driver_standings, on=['raceId', 'driverId'], how='left')
df = df.merge(constructor_standings, on=['raceId', 'constructorId'], how='left')

In [149]:
df.shape

(2138, 25)

In [150]:
drivers_red = drivers[['driverId', 'dob']]

In [151]:
df = df.merge(drivers_red, on='driverId', how='left')

In [152]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [153]:
results['positionOrder'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39])

In [154]:
results = results[results['raceId'] >= 1031]

In [155]:
results['positionOrder'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20])

In [156]:
results = results[['raceId', 'driverId', 'constructorId', 'positionOrder', 'positionText', 'rank']]
results = results.rename(columns={
    'positionText': 'result_position_text'
})

In [157]:
df = df.merge(results, on=['raceId', 'driverId', 'constructorId'], how='left')

In [158]:
df.shape

(2138, 29)

In [159]:
df.head()

,raceId,circuitId,year,round,name,date,time,qualifyId,driverId,constructorId,number,q1,q2,q3,grid,driverStandingsId,drv_points,drv_position,drv_position_text,drv_wins,constructorStandingsId,ctor_points,ctor_position,ctor_position_text,ctor_wins,dob,positionOrder,result_position_text,rank
0,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8378,822,131,77,1:04.111,1:03.015,1:02.939,1,69995,25.0,1,1,1,27573,37.0,1,1,1,1989-08-28,1,1,2
1,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8379,1,131,44,1:04.198,1:03.096,1:02.951,2,69998,12.0,4,4,0,27573,37.0,1,1,1,1985-01-07,4,4,3
2,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8380,830,9,33,1:04.024,1:04.000,1:03.477,3,70014,0.0,20,20,0,27581,0.0,9,9,0,1997-09-30,20,R,15
3,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8381,846,1,4,1:04.606,1:03.819,1:03.626,4,69997,16.0,3,3,0,27575,26.0,2,2,0,1999-11-13,3,3,1
4,1031,70,2020,1,Austrian Grand Prix,2020-07-05,13:10:00,8382,848,9,23,1:04.661,1:03.746,1:03.868,5,70007,0.0,13,13,0,27581,0.0,9,9,0,1996-03-23,13,13,7


In [160]:
df.dtypes

raceId                      int64
circuitId                   int64
year                        int64
round                       int64
name                       object
date                       object
time                       object
qualifyId                   int64
driverId                    int64
constructorId               int64
number                      int64
q1                         object
q2                         object
q3                         object
grid                        int64
driverStandingsId           int64
drv_points                float64
drv_position                int64
drv_position_text          object
drv_wins                    int64
constructorStandingsId      int64
ctor_points               float64
ctor_position               int64
ctor_position_text         object
ctor_wins                   int64
dob                        object
positionOrder               int64
result_position_text       object
rank                       object
dtype: object

In [161]:
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S')

In [162]:
df = df.replace("\\N", np.nan)
df.isna().sum()

raceId                       0
circuitId                    0
year                         0
round                        0
name                         0
date                         0
time                         0
qualifyId                    0
driverId                     0
constructorId                0
number                       0
q1                          21
q2                         551
q3                        1093
grid                         0
driverStandingsId            0
drv_points                   0
drv_position                 0
drv_position_text            0
drv_wins                     0
constructorStandingsId       0
ctor_points                  0
ctor_position                0
ctor_position_text           0
ctor_wins                    0
dob                          0
positionOrder                0
result_position_text         0
rank                         0
dtype: int64

In [163]:
print('year: ', df['year'].unique())
print('round: ', df['round'].unique())
print('name: ', df['name'].unique())
print('dob: ', df['dob'].unique())

year:  [2020 2021 2022 2023 2024]
round:  [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 20 18 19 21 22 23 24]
name:  ['Austrian Grand Prix' 'Styrian Grand Prix' 'Hungarian Grand Prix'
 'British Grand Prix' '70th Anniversary Grand Prix' 'Spanish Grand Prix'
 'Belgian Grand Prix' 'Italian Grand Prix' 'Tuscan Grand Prix'
 'Russian Grand Prix' 'Eifel Grand Prix' 'Portuguese Grand Prix'
 'Emilia Romagna Grand Prix' 'Turkish Grand Prix' 'Bahrain Grand Prix'
 'Sakhir Grand Prix' 'Abu Dhabi Grand Prix' 'Qatar Grand Prix'
 'Monaco Grand Prix' 'Azerbaijan Grand Prix' 'French Grand Prix'
 'Dutch Grand Prix' 'United States Grand Prix' 'Mexico City Grand Prix'
 'São Paulo Grand Prix' 'Saudi Arabian Grand Prix' 'Australian Grand Prix'
 'Miami Grand Prix' 'Canadian Grand Prix' 'Singapore Grand Prix'
 'Japanese Grand Prix' 'Las Vegas Grand Prix' 'Chinese Grand Prix']
dob:  <DatetimeArray>
['1989-08-28 00:00:00', '1985-01-07 00:00:00', '1997-09-30 00:00:00',
 '1999-11-13 00:00:00', '1996-03-23 00

Missing values in q1, q2 and q3 are actually not just a missing values. That is a feature. It is representing qualifications, in which step has driver came.

### Generating target value

In [164]:
print('Unique values for finish: ', df['result_position_text'].unique())

Unique values for finish:  ['1' '4' 'R' '3' '13' '6' '2' '5' '10' '7' '12' '8' '9' '11' '15' '16'
 '17' '14' '18' '19' 'W' '20' 'D']


In [165]:
df['final_position_num'] = pd.to_numeric(df['result_position_text'], errors='coerce')
print(df['final_position_num'].unique())

[ 1.  4. nan  3. 13.  6.  2.  5. 10.  7. 12.  8.  9. 11. 15. 16. 17. 14.
 18. 19. 20.]


In [166]:
df['scored_points'] = df['final_position_num'].apply(lambda x: 1 if x <= 10 else 0)
df['scored_points'].unique()

array([1, 0])

In [167]:
df['scored_points'].mean()

np.float64(0.500467726847521)

In [59]:
df.dtypes

raceId                             int64
circuitId                          int64
year                               int64
round                              int64
name                              object
date                      datetime64[ns]
time                      datetime64[ns]
qualifyId                          int64
driverId                           int64
constructorId                      int64
number                             int64
q1                                object
q2                                object
q3                                object
grid                               int64
driverStandingsId                  int64
drv_points                       float64
drv_position                       int64
drv_position_text                 object
drv_wins                           int64
constructorStandingsId             int64
ctor_points                      float64
ctor_position                      int64
ctor_position_text                object
ctor_wins       

# Model 1

### Libraries

In [53]:
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [82]:
# !pip install xgboost

In [54]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

### Eval functions

In [55]:
def evaluate_classifier(model_name, y_true, y_pred, y_proba=None):
    print('==========', model_name, '==========')

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1-score : {f1:.4f}')

    if y_proba is not None:
        roc = roc_auc_score(y_true, y_proba)
        print(f'ROC-AUC  : {roc:.4f}')

    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred), '\n')

In [56]:
def exp_win_eval(df, pipelines, features, target, year_col='year', evaluate_fn=None):
    years = sorted(df[year_col].unique())

    for i in range(1, len(years)):
        train_years = years[:i]
        test_year = years[i]

        train_df = df[df[year_col].isin(train_years)]
        test_df = df[df[year_col] == test_year]

        X_train = train_df[features]
        y_train = train_df[target]

        X_test = test_df[features]
        y_test = test_df[target]

        models = clone(pipelines)

        print(f"\n============= Iteration {i} | Train: {train_years} → Test: {test_year} =============")

        for name, model in models.items():
            print(f"\nTraining {name}...")

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            y_proba = None
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test)[:, 1]

            if evaluate_fn:
                evaluate_fn(model_name=name, y_true=y_test, y_pred=y_pred, y_proba=y_proba)

### Helper functions

In [57]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols, errors='ignore')

In [58]:
class QualiTimeConverter(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def _time_to_ms(self, t):
        if pd.isna(t):
            return None
        try:
            parts = t.split(':')
            minutes = int(parts[0])
            seconds = float(parts[1])
            total_ms = (minutes * 60 + seconds) * 1000
            return int(total_ms)
        except:
            return None 
    
    def transform(self, X):
        df = X.copy()

        for col in self.cols:
            new_col = f'{col}_ms'
            df[new_col] = df[col].apply(self._time_to_ms)
        
        return df

### Feature engineering

In [177]:
# Class which we use to create features that represent stats about driver before observed race

class DriverStandings(BaseEstimator, TransformerMixin):
    def __init__(
        self, 
        race_col='raceId', 
        driver_col='driverId', 
        season_col='year',
        round_col='round',
        points_col='drv_points', 
        position_col='drv_position', 
        wins_col='drv_wins'
    ):
        self.race_col=race_col
        self.driver_col=driver_col
        self.season_col=season_col
        self.round_col=round_col
        self.points_col=points_col
        self.position_col=position_col
        self.wins_col=wins_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df = df.sort_values(by=[self.driver_col, self.season_col, self.round_col])
        group_cols = [self.driver_col, self.season_col]
        
        df['before_race_points'] = df.groupby(group_cols)[self.points_col].shift(1)
        df['before_race_position'] = df.groupby(group_cols)[self.position_col].shift(1)
        df['before_race_wins'] = df.groupby(group_cols)[self.wins_col].shift(1)

        df.loc[df[self.round_col] == 1, [
            'before_race_points',
            'before_race_position',
            'before_race_wins'
        ]] = 0

        # if missing values
        df[['before_race_points', 'before_race_position', 'before_race_wins']] = \
            df[['before_race_points', 'before_race_position', 'before_race_wins']].fillna(0)

        return df

In [178]:
# Class which we use to create features that represent stats about constructor before observed race

class ConstructorStandings(BaseEstimator, TransformerMixin):
    def __init__(
        self, 
        race_col='raceId', 
        constructor_col='constructorId', 
        season_col='year',
        round_col='round',
        points_col='ctor_points', 
        position_col='ctor_position', 
        wins_col='ctor_wins'
    ):
        self.race_col=race_col
        self.constructor_col=constructor_col
        self.season_col=season_col
        self.round_col=round_col
        self.points_col=points_col
        self.position_col=position_col
        self.wins_col=wins_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df = df.sort_values(by=[self.constructor_col, self.season_col, self.round_col, self.race_col])
        group_cols = [self.constructor_col, self.season_col]

        df['cumulative_points'] = (
            df.groupby(group_cols)[self.points_col].cumsum()
        )
        
        df['before_race_ctor_points'] = df.groupby(group_cols)['cumulative_points'].shift(1).fillna(0)
        df['before_race_ctor_position'] = df.groupby(group_cols)[self.position_col].shift(1).fillna(0)
        df['before_race_ctor_wins'] = df.groupby(group_cols)[self.wins_col].shift(1).fillna(0)

        df.loc[df[self.round_col] == 1, [
            'before_race_ctor_points',
            'before_race_ctor_position',
            'before_race_ctor_wins'
        ]] = 0

        return df

### Preparing for train

In [136]:
y = 'scored_points'

cols_to_drop = [
    'raceId', 'circuitId', 'year', 'date', 'time', 'qualifyId', 'driverId', 'constructorId',
    'number', 'q1', 'q2', 'q3', 'grid', 'driverStandingsId', 'drv_points', 'drv_position', 
    'drv_position_text', 'drv_wins', 'constructorStandingsId', 'ctor_points', 'ctor_position', 
    'ctor_position_text', 'ctor_wins', 'dob', 'positionOrder', 'result_position_text', 'rank', 'final_position_num', 
    'scored_points'
]
               

numerical_cols = ['round']
categorical_cols = ['name']

after_engineering_cols = [
    'before_race_points', 'before_race_position', 'before_race_wins',
    'before_race_ctor_points', 'before_race_ctor_position', 'before_race_ctor_wins',
    'q1_ms', 'q2_ms', 'q3_ms'
]

In [137]:
feature_engineering_I = Pipeline(steps=[
    ('quali_covert', QualiTimeConverter(cols=['q1', 'q2', 'q3'])),
    ('drv_standings_before_race', DriverStandings()),
    ('ctor_standings_before_race', ConstructorStandings()),
    ('drop_cols', DropColumns(cols_to_drop))
])

In [138]:
numeric_transformer_I = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_I = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

column_transformer_I = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_I, numerical_cols + after_engineering_cols),
        ('cat', categorical_transformer_I, categorical_cols)
    ]
)

In [139]:
preprocessor_I = Pipeline(steps=[
    ('feature_engineering', feature_engineering_I),
    ('column_transformer', column_transformer_I)
])

In [140]:
pipeline_lr_I = Pipeline(steps=[
    ("preprocessing", preprocessor_I),
    ("model", LogisticRegression())
])

pipeline_rf_I =Pipeline(steps=[
    ("preprocessing", preprocessor_I),
    ("model", RandomForestClassifier())
])

pipeline_xgb_I =Pipeline(steps=[
    ("preprocessing", preprocessor_I),
    ("model", XGBClassifier())
])

In [141]:
pipelines_I = {
    "Logistic Regression": pipeline_lr_I,
    "Random Forest": pipeline_rf_I,
    "XGBoost": pipeline_xgb_I
}

### Train and test

In [142]:
features = cols_to_drop + numerical_cols + categorical_cols

In [35]:
exp_win_eval(
    df=df,
    pipelines=pipelines_I,
    features=features,
    target=y,
    evaluate_fn=evaluate_classifier
)

### Results

In [34]:
df_result_I = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [0.4943, 0.4874, 0.4942],
    "Precision": [0.4943, 0.4899, 0.4947],
    "Recall": [0.5239, 0.5156, 0.5153],
    "F1-score": [0.5060, 0.5018, 0.5030],
    "ROC-AUC": [0.4979, 0.4961, 0.5085]
})

df_result_I

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.4943,0.4943,0.5239,0.5060,0.4979
1,Random Forest,0.4874,0.4899,0.5156,0.5018,0.4961
2,XGBoost,0.4942,0.4947,0.5153,0.5030,0.5085


# Model 2

### Feature engineering

In [59]:
# Class that we will use to generate feature driver age.

class DriverAge(BaseEstimator, TransformerMixin):
    def __init__(self, date_col='date', dob_col='dob'):
        self.date_col = date_col
        self.dob_col = dob_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['driver_age'] = (df[self.date_col] - df[self.dob_col]).dt.days / 365.25
        return df

In [60]:
# Class that has method for creation race part of the day feature.

class PartOfDay(BaseEstimator, TransformerMixin):
    def __init__(self, time_col='time'):
        self.time_col = time_col

    def fit(self, X, y=None):
        return self

    def part_of_day(self, h):
        if 6 <= h < 12:
            return "morning"
        elif 12 <= h < 18:
            return "afternoon"
        else:
            return "evening"

    def transform(self, X):
        df = X.copy()
    
        df['hour'] = df[self.time_col].dt.hour
        df['race_part_of_day'] = df['hour'].apply(self.part_of_day)

        return df

In [61]:
# Class which has method for generating quali feature that tells us in which part of quali driver has finished.

class QualiRound(BaseEstimator, TransformerMixin):
    def __init__(self, q1_col='q1', q2_col='q2', q3_col='q3'):
        self.q1_col = q1_col
        self.q2_col = q2_col
        self.q3_col = q3_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X['quali_round'] = np.where(
            X[self.q2_col].isna(), 1,
            np.where(X[self.q3_col].isna(), 2, 3)
        )

        return X

### Preparing for train

In [79]:
y = 'scored_points'

cols_to_drop = [
    'raceId', 'circuitId', 'year', 'date', 'time', 'qualifyId', 'driverId', 'constructorId',
    'number', 'q1', 'q2', 'q3', 'grid', 'driverStandingsId', 'drv_points', 'drv_position', 
    'drv_position_text', 'drv_wins', 'constructorStandingsId', 'ctor_points', 'ctor_position', 
    'ctor_position_text', 'ctor_wins', 'dob', 'positionOrder', 'result_position_text', 'rank', 'final_position_num', 
    'scored_points'
]
               

numerical_cols = ['round']
categorical_cols = ['name']

after_engineering_num_cols = [
    'before_race_points', 'before_race_position', 'before_race_wins',
    'before_race_ctor_points', 'before_race_ctor_position', 'before_race_ctor_wins',
    'q1_ms', 'q2_ms', 'q3_ms', 'driver_age', 'quali_round'
]

after_engineering_cat_cols = ['race_part_of_day']

In [80]:
feature_engineering_II = Pipeline(steps=[
    ('quali_covert', QualiTimeConverter(cols=['q1', 'q2', 'q3'])),
    ('drv_standings_before_race', DriverStandings()),
    ('ctor_standings_before_race', ConstructorStandings()),
    ('driver_age', DriverAge()),
    ('race_part_of_day', PartOfDay()),
    ('quali_round', QualiRound()),
    ('drop_cols', DropColumns(cols_to_drop))
])

In [81]:
numeric_transformer_II = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_II = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

column_transformer_II = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_II, numerical_cols + after_engineering_num_cols),
        ('cat', categorical_transformer_II, categorical_cols + after_engineering_cat_cols)
    ]
)

In [82]:
preprocessor_II = Pipeline(steps=[
    ('feature_engineering', feature_engineering_II),
    ('column_transformer', column_transformer_II)
])

In [83]:
pipeline_lr_II = Pipeline(steps=[
    ("preprocessing", preprocessor_II),
    ("model", LogisticRegression())
])

pipeline_rf_II =Pipeline(steps=[
    ("preprocessing", preprocessor_II),
    ("model", RandomForestClassifier())
])

pipeline_xgb_II =Pipeline(steps=[
    ("preprocessing", preprocessor_II),
    ("model", XGBClassifier())
])

In [84]:
pipelines_II = {
    "Logistic Regression": pipeline_lr_II,
    "Random Forest": pipeline_rf_II,
    "XGBoost": pipeline_xgb_II
}

### Train and test

In [85]:
features = cols_to_drop + numerical_cols + categorical_cols

In [86]:
exp_win_eval(
    df=df,
    pipelines=pipelines_II,
    features=features,
    target=y,
    evaluate_fn=evaluate_classifier
)


============= Iteration 1 | Train: [2020] → Test: 2021 =============

Training Logistic Regression...
========== Logistic Regression ==========
Accuracy : 0.4806
Precision: 0.4785
Recall   : 0.4045
F1-score : 0.4384
ROC-AUC  : 0.4894
Confusion matrix:
[[122  97]
 [131  89]] 


Training Random Forest...
========== Random Forest ==========
Accuracy : 0.5125
Precision: 0.5161
Recall   : 0.4364
F1-score : 0.4729
ROC-AUC  : 0.5266
Confusion matrix:
[[129  90]
 [124  96]] 


Training XGBoost...
========== XGBoost ==========
Accuracy : 0.5080
Precision: 0.5085
Recall   : 0.5455
F1-score : 0.5263
ROC-AUC  : 0.5102
Confusion matrix:
[[103 116]
 [100 120]] 


============= Iteration 2 | Train: [2020, 2021] → Test: 2022 =============

Training Logistic Regression...
========== Logistic Regression ==========
Accuracy : 0.5341
Precision: 0.5304
Recall   : 0.5955
F1-score : 0.5610
ROC-AUC  : 0.5185
Confusion matrix:
[[104 116]
 [ 89 131]] 


Training Random Forest...
========== Random Forest ======

# Model 3

### Feature engineering

In [197]:
# Class that has method for creation quali gap to leaders

class QualiGap(BaseEstimator, TransformerMixin):
    def __init__(self, q2_col='q2_ms', q3_col='q3_ms', race_col='raceId'):
        self.q2_col = q2_col
        self.q3_col = q3_col
        self.race_col = race_col
        
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df['q3_gap_to_pole'] = df[self.q3_col] - df.groupby(self.race_col)[self.q3_col].transform('min')
        df['q2_gap_to_pole'] = df[self.q2_col] - df.groupby(self.race_col)[self.q2_col].transform('min')

        return df

In [218]:
# Class that creates column drivers form. Drivers form is average of n last races.

class AvgLastFinish(BaseEstimator, TransformerMixin):
    def __init__(self, year_col='year', round_col='round', driver_col='driverId', position_col='positionOrder', num_of_races=3):
        self.year_col = year_col
        self.round_col = round_col
        self.driver_col = driver_col
        self.position_col = position_col
        self.num_of_races = num_of_races

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df = df.sort_values(by=[self.year_col, self.round_col, self.driver_col])
        df[f'drv_avg_finish_last_{self.num_of_races}'] = df.groupby(self.driver_col)[self.position_col] \
            .shift(1).rolling(3).mean()

        return df

In [219]:
# Class that generates feature driver experience.

class DriverExperience(BaseEstimator, TransformerMixin):
    def __init__(self, driver_col='driverId'):
        self.driver_col = driver_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['driver_experience'] = df.groupby(self.driver_col).cumcount()
        return df

### Preparation pipelines

In [228]:
y = 'scored_points'

cols_to_drop = [
    'raceId', 'circuitId', 'year', 'date', 'time', 'qualifyId', 'driverId', 'constructorId',
    'number', 'q1', 'q2', 'q3', 'driverStandingsId', 'drv_points', 'drv_position', 
    'drv_position_text', 'drv_wins', 'constructorStandingsId', 'ctor_points', 'ctor_position', 
    'ctor_position_text', 'ctor_wins', 'dob', 'positionOrder', 'result_position_text', 'rank', 'final_position_num', 
    'scored_points'
]
               

numerical_cols = ['round', 'grid']
categorical_cols = ['name']

after_engineering_num_cols = [
    'before_race_points', 'before_race_position', 'before_race_wins',
    'before_race_ctor_points', 'before_race_ctor_position', 'before_race_ctor_wins',
    'q1_ms', 'q2_ms', 'q3_ms', 'driver_age', 'quali_round', 'q3_gap_to_pole', 'q2_gap_to_pole',
    'drv_avg_finish_last_3'
]

after_engineering_cat_cols = ['race_part_of_day']

In [229]:
feature_engineering_III = Pipeline(steps=[
    ('quali_covert', QualiTimeConverter(cols=['q1', 'q2', 'q3'])),
    ('drv_standings_before_race', DriverStandings()),
    ('ctor_standings_before_race', ConstructorStandings()),
    ('driver_age', DriverAge()),
    ('race_part_of_day', PartOfDay()),
    ('quali_round', QualiRound()),
    ('quali_gap', QualiGap()),
    ('avg_last_finish', AvgLastFinish()),
    ('driver_experience', DriverExperience()),
    ('drop_cols', DropColumns(cols_to_drop))
])

In [230]:
numeric_transformer_III = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_III = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

column_transformer_III = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_III, numerical_cols + after_engineering_num_cols),
        ('cat', categorical_transformer_III, categorical_cols + after_engineering_cat_cols)
    ]
)

In [231]:
preprocessor_III = Pipeline(steps=[
    ('feature_engineering', feature_engineering_III),
    ('column_transformer', column_transformer_III)
])

In [232]:
pipeline_lr_III = Pipeline(steps=[
    ("preprocessing", preprocessor_III),
    ("model", LogisticRegression())
])

pipeline_rf_III = Pipeline(steps=[
    ("preprocessing", preprocessor_III),
    ("model", RandomForestClassifier())
])

pipeline_xgb_III = Pipeline(steps=[
    ("preprocessing", preprocessor_III),
    ("model", XGBClassifier())
])

In [233]:
pipelines_III = {
    "Logistic Regression": pipeline_lr_III,
    "Random Forest": pipeline_rf_III,
    "XGBoost": pipeline_xgb_III
}

### Train and test

In [234]:
features = cols_to_drop + numerical_cols + categorical_cols

In [236]:
exp_win_eval(
    df=df,
    pipelines=pipelines_III,
    features=features,
    target=y,
    evaluate_fn=evaluate_classifier
)

### Results

In [237]:
df_results_III = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [0.7361, 0.7044, 0.6699],
    'Precision': [0.7503, 0.6885, 0.6643],
    'Recall': [0.7068, 0.7499, 0.6917],
    'F1-score': [0.7275, 0.7164, 0.6749],
    'ROC-AUC': [0.7921, 0.7631, 0.7286]
})

df_results_III

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.7361,0.7503,0.7068,0.7275,0.7921
1,Random Forest,0.7044,0.6885,0.7499,0.7164,0.7631
2,XGBoost,0.6699,0.6643,0.6917,0.6749,0.7286
